In [1]:
import pandas as pd

In [17]:
#Baseline stats
df = pd.read_csv("pet_clicks_sales.csv")
df['date'] = pd.to_datetime(df['date'])

In [21]:
import pandas as pd
from statsmodels.stats.power import TTestIndPower

metric = "leads"

mean = df[metric].mean()
std = df[metric].std()

effect_sizes = [0.05, 0.06, 0.07, 0.08, 0.09, 0.10]
powers = [0.70, 0.80, 0.90]

splits = {
    "50:50": 0.5,
    "40:60": 0.4,
    "30:70": 0.3,
    "20:80": 0.2,
    "10:90": 0.1,
}

analysis = TTestIndPower()

results = []

for split_name, treatment_share in splits.items():
    control_share = 1 - treatment_share
    ratio = treatment_share / control_share

    for lift in effect_sizes:
        absolute_effect = mean * lift
        cohen_d = absolute_effect / std

        for target_power in powers:
            n_control = analysis.solve_power(
                effect_size=cohen_d,
                alpha=0.05,
                power=target_power,
                ratio=ratio,
                alternative="two-sided"
            )

            n_treatment = n_control * ratio

            # calendar days are based on the larger group requirement
            calendar_days = max(
                n_control / control_share,
                n_treatment / treatment_share
            )

            results.append({
                "split": split_name,
                "power": f"{target_power:.0%}",
                "lift": f"{lift:.0%}",
                "mean_daily_leads": round(mean, 1),
                "daily_std": round(std, 1),
                "control_days_equiv": round(n_control),
                "treatment_days_equiv": round(n_treatment),
                "calendar_days_needed": round(calendar_days),
            })

power_table = pd.DataFrame(results)
power_table

,split,power,lift,mean_daily_leads,daily_std,control_days_equiv,treatment_days_equiv,calendar_days_needed
0,50:50,70%,5%,901.5,178.7,195,195,390
1,50:50,80%,5%,901.5,178.7,248,248,495
2,50:50,90%,5%,901.5,178.7,331,331,662
3,50:50,70%,6%,901.5,178.7,136,136,271
4,50:50,80%,6%,901.5,178.7,172,172,344
...,...,...,...,...,...,...,...,...
85,10:90,80%,9%,901.5,178.7,382,42,425
86,10:90,90%,9%,901.5,178.7,511,57,568
87,10:90,70%,10%,901.5,178.7,244,27,271
88,10:90,80%,10%,901.5,178.7,310,34,344


Power calculations - Safety

In [22]:
power_table.pivot_table(
    index=["split", "power"],
    columns="lift",
    values="calendar_days_needed"
)

lift           10%      5%      6%     7%     8%     9%
split power                                            
10:90 70%    271.0  1079.0   750.0  552.0  423.0  334.0
      80%    344.0  1372.0   953.0  701.0  537.0  425.0
      90%    461.0  1836.0  1276.0  938.0  718.0  568.0
20:80 70%    153.0   608.0   423.0  311.0  239.0  189.0
      80%    195.0   773.0   537.0  395.0  303.0  240.0
      90%    260.0  1034.0   718.0  528.0  405.0  320.0
30:70 70%    117.0   464.0   323.0  238.0  182.0  144.0
      80%    149.0   589.0   410.0  302.0  231.0  183.0
      90%    198.0   788.0   548.0  403.0  309.0  245.0
40:60 70%    103.0   406.0   283.0  208.0  160.0  127.0
      80%    130.0   516.0   359.0  264.0  203.0  161.0
      90%    174.0   690.0   480.0  353.0  271.0  214.0
50:50 70%     99.0   390.0   271.0  200.0  153.0  122.0
      80%    125.0   495.0   344.0  254.0  195.0  154.0
      90%    167.0   662.0   461.0  339.0  260.0  206.0